# CBOW — 4A, 4B, 4C, 4D

One CBOW Word2Vec model is reused for word embeddings, document classification and cosine-similarity recommendation.

In [27]:
import numpy as np
import pandas as pd
import re

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
# IMDB - 4A
data = pd.read_csv("IMDB Dataset.csv")
documents = data["review"].astype(str).tolist()
labels = data["sentiment"].map({"negative":0,"positive":1}).values

# 4C - Document
# documents = [
#     """The speed of transmission is an important difference between influenza and COVID-19. 
#     Influenza has a shorter incubation period and serial interval than COVID-19. 
#     The serial interval for COVID-19 is estimated to be 5-6 days, 
#     while for influenza it is 3 days. This means influenza can spread faster.

#     Transmission during the first 3-5 days of illness and pre-symptomatic 
#     transmission are major drivers of influenza spread. COVID-19 can also 
#     spread before symptoms appear, but this was considered a less significant 
#     driver in the passage.

#     The reproductive number for COVID-19 is estimated to be between 2 and 2.5, 
#     higher than influenza. However, estimates depend on context and time."""
# ]

# # 4B - Twitter, 4D - Recommendation 
# data = pd.read_csv("twitter_dataset.csv")
# documents = data["Text"].dropna().astype(str).tolist()

In [29]:
sentences = [re.findall(r"\b\w+\b",d.lower()) for d in documents]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(documents)

X = tokenizer.texts_to_sequences(documents)
X = pad_sequences(X)

print("Documents:",len(documents))
print("Vocabulary:",len(tokenizer.word_index))

Documents: 50000
Vocabulary: 124252


In [30]:
model = Sequential([
    Embedding(len(tokenizer.word_index)+1,50),
    GlobalAveragePooling1D(),
    Dense(50,activation="relu")
])

model.compile(optimizer="adam",loss="mse")

X = model.predict(X)

1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step


In [31]:
# 4A - IMDB
X_train,X_test,y_train,y_test = train_test_split(
    X,labels,test_size=0.2,random_state=42
)

clf = LogisticRegression(max_iter=500)
clf.fit(X_train,y_train)

print("Accuracy:",accuracy_score(y_test,clf.predict(X_test)))

Accuracy: 0.4961


In [32]:
# # # 4B , 4C - Twitter
# word = list(tokenizer.word_index.keys())
# print("Word:",words)

# 4D - Recommendation
# words = tokenizer.word_index
# similarity = cosine_similarity(
#     model.layers[0].get_weights()[0][words[word]].reshape(1,-1),
#     model.layers[0].get_weights()[0]
# )[0]

# for i in similarity.argsort()[::-1][1:6]:
#     print(list(words.keys())[list(words.values()).index(i+1)])

In [33]:
# query = input("Enter search text: ")

# q = pad_sequences(
#     tokenizer.texts_to_sequences([query]),
#     maxlen=X.shape[1]
# )

# q = model.predict(q)

# score = cosine_similarity(q,X)[0]

# for i in score.argsort()[::-1][:5]:
#     print(round(float(score[i]),3),documents[i][:200])

In [34]:
!pip install nltk

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 11.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Load data
df = pd.read_csv("IMDB Dataset.csv")
texts = df['Text'].fillna('').tolist() if 'Text' in df else df.iloc[:,0].fillna('').tolist()
texts = [re.sub(r'[^a-z ]', '', t.lower()) for t in texts]

# Tokenization
tok = Tokenizer()
tok.fit_on_texts(texts)
X = pad_sequences(tok.texts_to_sequences(texts), maxlen=100)
v = len(tok.word_index) + 1

# CBOW-style model
model = Sequential([
    Embedding(v, 50),
    GlobalAveragePooling1D(),
    Dense(32, activation='relu')
])

# Classification (4A)
label = next((c for c in ['review','sentiment'] if c in df.columns), None)

if label:
    y = LabelEncoder().fit_transform(df[label].astype(str))
    a,b,c,d = train_test_split(X,y,test_size=0.2,random_state=42)

    clf = Sequential([
        Embedding(v,50),
        GlobalAveragePooling1D(),
        Dense(32,activation='relu'),
        Dense(len(np.unique(y)),activation='softmax')
    ])
    clf.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    clf.fit(a,c,epochs=5)
    print("Test accuracy:",clf.evaluate(b,d)[1])

# Document vectors (4B, 4C, 4D)
model.compile(optimizer='adam',loss='mse')
vectors = model.predict(X,verbose=0)

# Word embedding
print("Word embedding:",model.layers[0].get_weights()[0][1])

# Recommendation
q = input("Enter text: ")
q = pad_sequences(tok.texts_to_sequences([q]),maxlen=100)
sim = cosine_similarity(model.predict(q,verbose=0),vectors)[0]

for i in sim.argsort()[-5:][::-1]:
    print(texts[i],round(sim[i],3))

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 114s 89ms/step - accuracy: 0.0000e+00 - loss: 10.9856
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 114s 91ms/step - accuracy: 0.0000e+00 - loss: 10.2139
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 111s 89ms/step - accuracy: 1.0000e-04 - loss: 10.1586
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 114s 91ms/step - accuracy: 1.0000e-04 - loss: 9.7726
Epoch 5/5
1001/1250 ━━━━━━━━━━━━━━━━━━━━ 22s 88ms/step - accuracy: 1.5609e-04 - loss: 10.0900